In [1]:

# Numerical libraries
import numpy as np
import random as rnd

# Machine learning and data preprocessing libraries
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Plotting library
from keras.preprocessing.sequence import pad_sequences
import matplotlib.pyplot as plt

import os, sys, datetime
os.environ['TF_CPP_MIN_LOG_LEVEL']='2'

# PCC model
from kinematics_functions import T_beModule, invKspace_car
# from point_clouds import generate_random_points
# from add_error_to_kin import add_error_to_kin
# from ISTlogo_traj import ISTlogo_traj

# read data from Polaris dataset
from data_functions import parse_dataset, polaris2base

# import tensorflow.compat.v1 as tf
# tf.enable_eager_execution(tf.ConfigProto(log_device_placement=False)) 


# config = tf.ConfigProto(device_count = {'GPU': 1})


In [2]:
rnn_flag = False
rnn_flag = True

diff_flag = False
# diff_flag = True

diff_data = 'cable'
diff_data = 'xyz'

neib_flag = False
# neib_flag = True

write_flag = False
write_flag = True

# data_size_range = np.linspace(0.1, 1, 5)
data_size_range = np.linspace(0.125, 1, 8)
# data_size_range = np.linspace(0.125, .875, 4)
# data_size_range = np.linspace(0.25, 1, 4)
# data_size_range = [0.125, 0.50, 1]
# data_size_range = [1]
test_split = 0.15

# n_range = range(20, 70 + 1, 10)
# n_range = [20, 30, 60, 70]
# h_range = range(4, 4 + 1)
n_range = range(25, 65 + 1, 10)
h_range = range(1, 3 + 1)


MAX_EPOCH = 6000
# MAX_EPOCH = 500
PATIENCE = 300
# PATIENCE = -1
BATCH_SIZE = 128
ACTIVATION_FUN = 'selu'
ACTIVATION_FUN = 'tanh'
ACTIVATION_FUN = 'sigmoid'
ACTIVATION_FUN = 'elu'
ACTIVATION_FUN = 'relu'

In [3]:

if neib_flag:
    # random w/ neighbors
    datasets = [
        "./data/dataset_1051_rand3N_2024-02-07T114202.txt", # ~500
        "./data/dataset_1051_rand3N_2024-02-07T124945.txt",
        "./data/dataset_1049_rand3N_2024-02-07T144446.txt",
        "./data/dataset_1047_rand3N_2024-02-07T155230.txt", # ~900
        "./data/dataset_7687_rand3N_2024-02-09T113646.txt", # also aprox 1100
        "./data/dataset_1047_rand3N_2024-02-09T130942.txt",
        "./data/dataset_841_rand3N_2024-02-09T164215.txt",
        "./data/dataset_1201_2023-12-19T155359.txt", # for testing, random
        ]

else:
    # random datasets
    datasets = [
        "./data/dataset_1200_2023-12-13T161233.txt",
        "./data/dataset_1200_2023-12-15T145833.txt", # many outliers
        "./data/dataset_1201_2023-12-19T155359.txt",
        "./data/dataset_1100_2024-02-07T165437.txt", # without extreme outlier
        "./data/dataset_1101_2024-02-09T103741.txt",
        "./data/dataset_1101_2024-02-09T141149.txt", # ~700
        ]

jtheta2len = lambda p: (p - 4488.62157) / -26.03760

datasets_result = []

for filename in datasets:
    traj, ref_T, pos = parse_dataset(filename)
    traj_, pos_base, est_pos_base = polaris2base(traj, ref_T, pos)
    datasets_result.append({"traj": traj, "traj_": traj_, "pos_base": pos_base, "est_pos_base": est_pos_base})

x, y = [], []
x_test, y_test = [], []

rnd.seed(420)
inputs = []
outputs = []

num_time_steps = 2
min_max_scaler = MinMaxScaler()
x = [min_max_scaler.fit_transform(seq) for seq in x]

test_idx = rnd.randint(0, len(datasets_result))
# test_idx = np.random.randint(0,len(datasets_result), seed)
# x_test = datasets_result[test_idx]["pos_base"]
x_test = np.array([invKspace_car(*p, theta_flag=False) for p in datasets_result[test_idx]["pos_base"]])
test_inputs = [tuple(x_test[i:i+num_time_steps,:]) for i in range(1, len(x_test) - num_time_steps + 1)]
y_test = datasets_result[test_idx]["traj_"]
test_outputs = [y_test[i+num_time_steps-1,:] for i in range(1, len(y_test) - num_time_steps + 1)]
datasets_result.pop(test_idx)
x = []
# for dataset in datasets_result:
#     x += [invKspace_car(*p, theta_flag=False) for p in dataset.get("pos_base")]
x = [np.array([invKspace_car(*p, theta_flag=False) for p in dataset.get("pos_base")]) for dataset in datasets_result]
y = [dataset.get("traj_", []) for dataset in datasets_result]


for j in range(len(x)):
    inputs += [tuple(x[j][i:i+num_time_steps,:]) for i in range(1, len(x[j]) - num_time_steps + 1)]
    outputs += [y[j][i+num_time_steps-1,:] for i in range(1, len(x[j]) - num_time_steps + 1)]
    # for i in range(1, len(x[j]) - 1):
    #     inputs.append(tuple(x[j][i:i+2,:]))  # Extract positions for two consecutive time steps
    #     outputs.append(y[j][i+1,:])  # Joint angles at the second time step
        # segments.append((input_pair, output_joint_angles))

inputs = np.array(inputs)
outputs = np.array(outputs)
test_inputs = np.array(test_inputs)
test_outputs = np.array(test_outputs)
print(test_idx)
print(inputs.shape)
print(outputs.shape)
print(test_inputs.shape)
print(test_outputs.shape)

# x_test = min_max_scaler.fit_transform(x_test) # pos

# print(x_test.shape, y_test.shape)


0
(5324, 2, 3)
(5324, 3)
(1198, 2, 3)
(1198, 3)


In [4]:
dt = datetime.datetime.now(datetime.timezone.utc).isoformat().split('.')[0].replace(':', '')

if write_flag:
    results_folder = "./results/final_datasets_rnn_comp"
    if not os.path.exists(results_folder): os.makedirs(results_folder)
    if diff_flag:
        if neib_flag: result_file = open(f'{results_folder}/results_diff_rand3N_{dt}.txt', 'w')
        else: result_file = open(f'{results_folder}/results_diff_rand_{dt}.txt', 'w')
    else: 
        if neib_flag: result_file = open(f'{results_folder}/results_rand3N_{dt}.txt', 'w')
        else: result_file = open(f'{results_folder}/results_rand_{dt}.txt', 'w')
    result_file.write(f'Train Set Size: {len(x)}\n')
    result_file.write(f'Test Set Size: {len(x_test)}\n')
    result_file.write(f'NN Params: Max Epochs - {MAX_EPOCH}; Activation Function - \'{ACTIVATION_FUN}\'; Stopping Criteria Patience - {PATIENCE}; Train Data Shape- {inputs.shape}\n')
    # final_datasets2
    # result_file.write(f'\nArchitechture;pct of train dataset;Epochs;Last Val Loss;MRE [%];MaxRE [%];MAE [mm];MaxAE [mm];MRE (Mean Length) [%];MaxRE (Mean Length) [%];MRE (Full-Scale) [%];MaxRE (Full-Scale) [%]')
    # final_datasets3
    result_file.write(f'\nArchitechture;pct of train dataset;Epochs;Last Val Loss;MRE [%];MaxRE [%];MAE [mm];MaxAE [mm];StdAE [mm];Mean Length;Full-Scale')
    result_file.flush()

In [7]:
%reload_ext autoreload
%autoreload 2

from nn_builder import DNNModelBuilder
from nn_builder import mlp_1, rnn, SimpleRNN

if write_flag:
    models_folder = f"{results_folder}/models_{dt}"
    if not os.path.exists(models_folder): os.makedirs(models_folder)

# max_len = max([len(seq) for seq in x] + [len(x_test)])
for data_size in data_size_range:
    if data_size != 1:
        # x_train, _, y_train, _ = train_test_split(x, y, test_size=1-data_size, random_state=93216)
        x_split = int(len(inputs) * data_size)
        y_split = int(len(outputs) * data_size)
        assert(x_split == y_split)
        x_train = inputs[:x_split]
        y_train = outputs[:y_split]
    else: x_train, y_train = inputs, outputs
        
    # xyz_diff = min_max_scaler.fit_transform(np.array(xyz_diff))
    # x_train = min_max_scaler.fit_transform(x_train) # pos

    # ## Model Design & Selection
    models = []
    for h in h_range:
        if rnn_flag and h > 1: models =  [(rnn, [[(n, True)]*(h-2) + [(n, False)]*2], {}) for n in n_range] # eg
        elif rnn_flag: models =  [(rnn, [[(n, False)]*h], {}) for n in n_range] # eg
        # if rnn_flag: models =  [(SimpleRNN, [], {}) for n in n_range] # eg
        else: models =  [(mlp_1, [[n]*h], {'activation': ACTIVATION_FUN}) for n in n_range] # eg

    # Using the model builder
        builder = DNNModelBuilder(x_train, models)

        # continue
        # print(f"Last model in step: {models[-1][1]}")
        # print(builder.models[-1].summary())

        # print(x_train.shape)
        # if rnn_flag:
        #     # for fold in range(len(x_train)):
        #     #     x_val = x_train[fold:fold+1]
        #     #     y_val = y_train[fold:fold+1]
                
        #     #     # Use all other sequences for training
        #     #     x_fold = np.concatenate([x_train[:fold], x_train[fold+1:]], axis=0)
        #     #     y_fold = np.concatenate([y_train[:fold], y_train[fold+1:]], axis=0)
        #     #     builder.train(x_fold, y_fold, epochs=MAX_EPOCH, batch_size=BATCH_SIZE, patience=PATIENCE, validation_data=(x_val, y_val))
            
        #     # x_train = x_train.reshape(1, sum([len(seq) for seq in x_train]), x_train.shape[2])
        #     # y_train = y_train.reshape(1, sum([len(seq) for seq in y_train]), y_train.shape[2])
        #     # builder.train(x_train, y_train, epochs=MAX_EPOCH, batch_size=BATCH_SIZE, patience=PATIENCE, validation_split=0)

        #     builder.train(x_train, y_train, epochs=MAX_EPOCH, batch_size=BATCH_SIZE, patience=PATIENCE)
        # else:
        builder.train(x_train, y_train, epochs=MAX_EPOCH, batch_size=BATCH_SIZE, validation_split=test_split/(1 - test_split), patience=PATIENCE)
        # for i, model in enumerate(builder.architectures):
        #     print(f"Model {i+1}: {str(model[0]).split(' ')[1]}, {model[1][0]}")
        builder.compare_models(stack=False)
        # losses = builder.evaluate(x_test, y_test)
        # builder.save_models()

        pred = builder.predict(test_inputs)

        # print("Mean Lengths:", np.mean(y_test))
        y_test = test_outputs
        for i, p in enumerate(pred):
            # plot_positions_comparison(np.array(y_test), np.array(p))
            pred_abs_norm_err = np.abs(np.linalg.norm(p - y_test, axis=1))
            pred_rel_mean_err = pred_abs_norm_err / np.mean(y_test) * 100
            pred_rel_norm_err = np.abs(np.linalg.norm(p - y_test, axis=1)) / np.abs(np.linalg.norm(y_test, axis=1)) * 100
            pred_rel_scale_err = pred_abs_norm_err / (np.max(y_test[:,0]) - np.min(y_test[:,0])) * 100
            print(f"\nArchitechture: {str(builder.architectures[i][0]).split(' ')[1]}, {builder.architectures[i][1][0]}")
            print(f"% of data: {data_size * 100:.2f}%")
            print(f"Mean Relative Error: {np.mean(pred_rel_norm_err):.3f}%")
            print(f"Max Relative Error: {max(pred_rel_norm_err):.3f}%")
            print(f"Mean Absolute Error: {np.mean(pred_abs_norm_err):.3f} mm")
            print(f"Max Absolute Error: {max(pred_abs_norm_err):.3f} mm")
            # print(f"Mean Relative Error (Mean Length): {np.mean(pred_rel_mean_err):.3f} %")
            # print(f"Max Relative Error (Mean Length): {max(pred_rel_mean_err):.3f} %")
            print(f"Mean Relative Error (Full-Scale): {np.mean(pred_rel_scale_err):.3f}%")
            print(f"Max Relative Error (Full-Scale): {max(pred_rel_scale_err):.3f}%")

            if write_flag:
                result_file.write(f"\n{str(builder.architectures[i][0]).split(' ')[1]}, {builder.architectures[i][1][0]}; ")
                result_file.write(f"{data_size*100:.2f}; ")
                if builder.early_stopping == None or builder.early_stopping.stopped_epoch == None or builder.early_stopping.stopped_epoch == 0:
                    result_file.write(f"{MAX_EPOCH}; ")
                else: result_file.write(f"{len(builder.histories[i].history['val_loss'])}; ")
                result_file.write(f"{builder.histories[i].history['val_loss'][-1]:3f}; ")
                result_file.write(f"{np.mean(pred_rel_norm_err):.3f}; ")
                result_file.write(f"{max(pred_rel_norm_err):.3f}; ")
                result_file.write(f"{np.mean(pred_abs_norm_err):.3f}; ")
                result_file.write(f"{max(pred_abs_norm_err):.3f}; ")
                
                result_file.write(f"{np.std(pred_abs_norm_err):.3f}; ")
                result_file.write(f"{np.mean(y_test):3f}; ")
                result_file.write(f"{np.max(y_test[:,0]) - np.min(y_test[:,0]):3f}")
                
                # result_file.write(f"{np.mean(pred_rel_mean_err):.3f}; ")
                # result_file.write(f"{max(pred_rel_mean_err):.3f}; ")
                # result_file.write(f"{np.mean(pred_rel_scale_err):.3f}; ")
                # result_file.write(f"{max(pred_rel_scale_err):.3f}")
                
                result_file.flush()

                builder.save_models(prefix=f"{str(builder.architectures[0][0]).split()[1]}_{int(data_size*100)}pct_{h}L", folder=models_folder)

                if not os.path.exists(f"{models_folder}/imgs/"): os.makedirs(f"{models_folder}/imgs/")
        
            if write_flag and data_size in [0.25, 0.5, 0.75, 1]: plt.savefig(f"{models_folder}/imgs/{str(builder.architectures[0][0]).split()[1]}_{int(data_size*100)}pct_{h}L.png")
            # if write_flag and data_size in [0.25, 0.5, 1]: plt.savefig(f'./results/final_datasets2/imgs/training_{data_size*100:.0f}_{h}L_{dt}.png')

if write_flag: result_file.close()


In [6]:
# if not os.path.exists(f"{folder}/imgs/"): os.makedirs(f"{folder}/imgs/")
# print(test_outputs.shape)
# builder.predict(x_test)
# plt.savefig(f"{folder}/imgs/{str(builder.architectures[0][0]).split()[1]}_{int(data_size*100)}pct_{h}L.png")

# result_file.flush()

# result_file.close()